## 透视表数据背景

我们有一个 **宽表**（每行一个学生，各科成绩分别占一列）：

| 姓名    | 数学 | 英语 | 语文 |
| ----- | -- | -- | -- |
| Mary  | 97 | 89 | 93 |
| David | 88 | 79 | 98 |

接下来所有讲解 **只围绕这个宽表和它转换后的长表**。

---

##  1. 什么是宽表（Wide）

宽表特点：

* 每个变量占一列（数学/英语/语文）
* 每人占一行
* 列比较多

结构：

```
一行 = 一个学生
一列 = 一门课程的成绩
```

---

## 2. 什么是长表（Long）

长表把“列的变量”变成“行的变量”：

| 姓名    | 课程 | 成绩 |
| ----- | -- | -- |
| Mary  | 数学 | 97 |
| Mary  | 英语 | 89 |
| Mary  | 语文 | 93 |
| David | 数学 | 88 |
| David | 英语 | 79 |
| David | 语文 | 98 |

结构：

```
一行 = 一个学生的一门课成绩
变量名（数学/英语/语文）变成了一列的数据
```

---

## 3. 宽表 → 长表（melt）的原理是什么？

**核心本质只有一句话：**

> 把“多列变量（数学/英语/语文）”拆成“变量名列（课程）+ 变量值列（成绩）”。

例如 Mary 的一行：

```
数学 97 → 课程=数学，成绩=97
英语 89 → 课程=英语，成绩=89
语文 93 → 课程=语文，成绩=93
```

所以宽表的一行
→ 长表的三行。

没有任何额外复杂逻辑，就是“拆列变行”。

---

##  4. 长表 → 宽表（pivot）的原理是什么？

**核心本质一句话：**

> 把“课程列中的类别（数学/英语/语文）”展开成多列。

例如：

课程列里出现三个类别：

```
数学
英语
语文
```

pivot 会把它们“横向展开”成 3 列，并填入成绩。

长表三行
→ 宽表一行。

---

##  5. 透视表 pivot_table 的本质

pivot_table = pivot + 自动统计

例如：求每人每科平均成绩（其实就是恢复宽表的形式）

它的结构依然是：

* index：行标签（例如姓名）
* columns：列标签（例如课程）
* values：要填的数（成绩）
* aggfunc：怎么计算（平均 / 求和）

本质与 pivot 一样，只是更灵活。

---

# 透视表的语法
pd.pivot_table(data, index, columns, values, aggfunc, margins, normalize)
- data: dataframe数据
- index: 透视表的行名
- columns: 透视表的列名
- values: 透视表需要计算的数据列
- aggfunc: 对values数据列使用的计算函数，一般用numpy的计算函数，如np.mean, np.sum 等
- margins: 是否显示汇总信息，默认margins=False
- normalize: 显示百分比，可选'all','index','columns'参数值

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('table.csv')
df.head()

,School,Class,ID,Gender,Address,Height,Weight,Math,Physics
0,S_1,C_1,1101,M,street_1,173,63,34.0,A+
1,S_1,C_1,1102,F,street_2,192,73,32.5,B+
2,S_1,C_1,1103,M,street_2,186,82,87.2,B+
3,S_1,C_1,1104,F,street_2,167,81,80.4,B-
4,S_1,C_1,1105,F,street_4,159,64,84.8,B+


In [8]:
# 宽表 → 长表（生成课程/成绩列）
df_long = pd.melt(
    df,
    id_vars=['ID'],               # 主体列
    value_vars=['Math'],  # 要拆的课程列，根据你的 df 实际情况来
    var_name='课程',
    value_name='成绩'
)
print(df_long)

      ID    课程    成绩
0   1101  Math  34.0
1   1102  Math  32.5
2   1103  Math  87.2
3   1104  Math  80.4
4   1105  Math  84.8
5   1201  Math  97.0
6   1202  Math  63.5
7   1203  Math  58.8
8   1204  Math  33.8
9   1205  Math  68.4
10  1301  Math  31.5
11  1302  Math  87.7
12  1303  Math  49.7
13  1304  Math  85.2
14  1305  Math  61.7
15  2101  Math  83.3
16  2102  Math  50.6
17  2103  Math  52.5
18  2104  Math  72.2
19  2105  Math  34.2
20  2201  Math  39.1
21  2202  Math  68.5
22  2203  Math  73.8
23  2204  Math  47.2
24  2205  Math  85.4
25  2301  Math  72.3
26  2302  Math  32.7
27  2303  Math  65.9
28  2304  Math  95.5
29  2305  Math  48.9
30  2401  Math  45.3
31  2402  Math  48.7
32  2403  Math  59.7
33  2404  Math  67.7
34  2405  Math  47.6


In [14]:
# 目标：计算“每个学校 × 每种性别”的平均体重
#把 df 这张学生表，按照 学校 分行、按照 性别 分列，对每一格里的所有学生 体重求平均值。
pd.pivot_table(
    df,              # data：原始数据表，每一行是一名学生
    index='School',  # index：透视表的“行标签”
                     #        这里按学校分行，每一行是一个 School（比如 S_1、S_2）

    columns='Gender',# columns：透视表的“列标签”
                     #          这里按性别分列，一般是 'M'（男）、'F'（女）

    values='Weight', # values：要进行计算的那一列
                     #         这里用 Weight 列，表示学生的体重

    aggfunc='mean'  # aggfunc：对 values 怎么算？这里用 np.mean 求“平均值”
                     #         即：同一学校 + 同一性别的所有学生体重，求平均
)

Gender,F,M
School,,
S_1,70.500000,69.428571
S_2,75.363636,81.555556
